# NYC 311 Data Cleaning Notebook (2022-2025)

This notebook builds a reproducible cleaning pipeline for NYC 311 data and adds targeted EDA checks needed for cleaning decisions beyond the profiling report.

Outputs:

- `data/processed/311_clean_2022_2025.parquet`
- `data/processed/311_data_quality_log_2022_2025.csv`


## 1) Setup and Imports

Load core libraries needed for deterministic cleaning and data-quality logging.


In [5]:
import json
import uuid
from pathlib import Path
import pandas as pd

pd.set_option("display.max_columns", 120)

## 2) Paths and Output Targets

Define project-relative paths for raw input and processed outputs.

This section also creates the processed directory if it does not exist.


In [6]:
PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / "data").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent

RAW_PATH = (
    PROJECT_ROOT
    / "data"
    / "raw"
    / "311_Service_Requests_from_2020_to_Present_20260502.csv"
)
PROCESSED_DIR = PROJECT_ROOT / "data" / "processed"
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

CLEAN_PARQUET = PROCESSED_DIR / "311_clean_2022_2025.parquet"
DQ_LOG_CSV = PROCESSED_DIR / "311_data_quality_log_2022_2025.csv"
PIPELINE_DATE = pd.Timestamp("2026-05-02")
RUN_ID = str(uuid.uuid4())[:8]
SUMMARY_JSON = PROCESSED_DIR / "pipeline_summary.json"

RAW_PATH

PosixPath('/Users/tuanhung/Documents/GitHub/NYC311/data/raw/311_Service_Requests_from_2020_to_Present_20260502.csv')

## 3) Load Raw Dataset

Read the NYC 311 raw file using string dtypes first so we can apply controlled parsing and cleaning rules.


In [7]:
df = pd.read_csv(RAW_PATH, dtype=str, low_memory=False)
print("raw shape:", df.shape)
df.head(2)

raw shape: (9862266, 44)


,Unique Key,Created Date,Closed Date,Agency,Agency Name,Problem (formerly Complaint Type),Problem Detail (formerly Descriptor),Additional Details,Location Type,Incident Zip,Incident Address,Street Name,Cross Street 1,Cross Street 2,Intersection Street 1,Intersection Street 2,Address Type,City,Landmark,Facility Type,Status,Due Date,Resolution Description,Resolution Action Updated Date,Community Board,Council District,Police Precinct,BBL,Borough,X Coordinate (State Plane),Y Coordinate (State Plane),Open Data Channel Type,Park Facility Name,Park Borough,Vehicle Type,Taxi Company Borough,Taxi Pick Up Location,Bridge Highway Name,Bridge Highway Direction,Road Ramp,Bridge Highway Segment,Latitude,Longitude,Location
0,52940375,01/01/2022 12:00:00 AM,01/03/2022 08:39:00 AM,DEP,Department of Environmental Protection,Air Quality,"Air: Odor/Fumes, Vehicle Idling (AD3)",AD3,NaN,10036,640 8 AVENUE,8 AVENUE,W 41 ST,W 42 ST,NaN,NaN,ADDRESS,NEW YORK,NaN,NaN,Closed,NaN,The Department of Environmental Protection det...,01/03/2022 08:39:00 AM,05 MANHATTAN,03,Precinct 14,1010137501,MANHATTAN,"986,967","214,950",ONLINE,Unspecified,MANHATTAN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,40.75666417743,-73.99019293432,POINT (-73.990192934325 40.756664177427)
1,52934953,01/01/2022 12:00:10 AM,01/01/2022 01:00:11 AM,NYPD,New York City Police Department,Noise - Street/Sidewalk,Loud Music/Party,NaN,Street/Sidewalk,11222,126 DRIGGS AVENUE,DRIGGS AVENUE,NORTH HENRY STREET,RUSSELL STREET,NORTH HENRY STREET,RUSSELL STREET,ADDRESS,BROOKLYN,DRIGGS AVENUE,NaN,Closed,NaN,The Police Department responded to the complai...,01/01/2022 01:00:15 AM,01 BROOKLYN,33,Precinct 94,NaN,BROOKLYN,"999,866","202,742",MOBILE,Unspecified,BROOKLYN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,40.72314288436,-73.94366208446,POINT (-73.943662084458 40.723142884361)


## 4) Standardize Key Fields and Parse Timestamps

Map key source columns to cleaning-friendly aliases and parse `Created Date`/`Closed Date` into datetime fields with fallback parsing logic.


In [8]:
colmap = {
    "Unique Key": "unique_key",
    "Created Date": "created_date_raw",
    "Closed Date": "closed_date_raw",
    "Agency": "agency",
    "Agency Name": "agency_name",
    "Problem (formerly Complaint Type)": "complaint_type",
    "Problem Detail (formerly Descriptor)": "descriptor",
    "Borough": "borough",
    "Incident Zip": "incident_zip",
    "Open Data Channel Type": "open_data_channel_type",
    "Status": "status",
}

missing_required = [c for c in colmap if c not in df.columns]
if missing_required:
    raise KeyError(f"Missing expected columns: {missing_required}")

for src, dst in colmap.items():
    df[dst] = df[src]


def parse_dt(s: pd.Series) -> pd.Series:
    s = s.astype("string").str.strip()
    s = s.replace({"": pd.NA, "nan": pd.NA, "None": pd.NA})
    return pd.to_datetime(s, format="%m/%d/%Y %I:%M:%S %p", errors="coerce")


df["created_ts"] = parse_dt(df["created_date_raw"])
df["closed_ts"] = parse_dt(df["closed_date_raw"])

## 5) Data Quality Log Framework

Create a reusable logging helper so every exclusion/check is documented with counts, rule, severity, and action.


In [9]:
dq_records = []


def log_check(check_name, failed_mask, severity="error", rule="", action="", notes=""):
    failed = int(failed_mask.sum())
    total = len(failed_mask)
    dq_records.append(
        {
            "check_name": check_name,
            "severity": severity,
            "failed_rows": failed,
            "total_rows": total,
            "failed_pct": round(100 * failed / total, 4) if total else 0.0,
            "rule": rule,
            "action": action,
            "notes": notes,
            "run_ts": pd.Timestamp.utcnow(),
        }
    )


log_check(
    "created_ts_parse_failed",
    df["created_ts"].isna(),
    severity="error",
    rule="Created Date must parse to datetime",
    action="drop",
)

log_check(
    "closed_ts_parse_failed_when_present",
    df["closed_date_raw"].notna()
    & df["closed_date_raw"].astype(str).str.strip().ne("")
    & df["closed_ts"].isna(),
    severity="warn",
    rule="Closed Date must parse when non-empty",
    action="retain_open_flag_or_drop_if_invalid",
)

pd.DataFrame(dq_records)

,check_name,severity,failed_rows,total_rows,failed_pct,rule,action,notes,run_ts
0,created_ts_parse_failed,error,0,9862266,0.0,Created Date must parse to datetime,drop,,2026-05-06 12:37:23.217431+00:00
1,closed_ts_parse_failed_when_present,warn,0,9862266,0.0,Closed Date must parse when non-empty,retain_open_flag_or_drop_if_invalid,,2026-05-06 12:37:27.167314+00:00


## 6) Cleaning-Critical EDA Checks

Run project-specific checks that profiling alone does not finalize:

- enforce the 2022-2025 analytical window
- identify negative durations and implausible closure timestamps
- quantify duplicate keys/rows


In [10]:
start = pd.Timestamp("2022-01-01")
end = pd.Timestamp("2025-12-31 23:59:59")

mask_slice = df["created_ts"].between(start, end, inclusive="both")
log_check(
    "created_outside_2022_2025",
    (~mask_slice) | df["created_ts"].isna(),
    severity="info",
    rule="Created timestamp in 2022-01-01..2025-12-31",
    action="drop",
)

dfw = df.loc[mask_slice].copy()
print("slice shape:", dfw.shape)

dfw["is_closed"] = dfw["closed_ts"].notna()
dfw["resolution_minutes"] = (
    dfw["closed_ts"] - dfw["created_ts"]
).dt.total_seconds() / 60.0

neg_duration = dfw["is_closed"] & (dfw["resolution_minutes"] < 0)
log_check(
    "negative_resolution_duration",
    neg_duration,
    severity="error",
    rule="closed_ts >= created_ts for closed cases",
    action="drop",
)

closed_outlier = dfw["is_closed"] & (
    (dfw["closed_ts"] < pd.Timestamp("2020-01-01"))
    | (dfw["closed_ts"] > PIPELINE_DATE + pd.Timedelta(days=30))
)
log_check(
    "implausible_closed_ts",
    closed_outlier,
    severity="error",
    rule="closed_ts in [2020-01-01, today+30d]",
    action="drop",
)

dup_unique_key = dfw["unique_key"].duplicated(keep=False)
log_check(
    "duplicate_unique_key_in_slice",
    dup_unique_key,
    severity="error",
    rule="Unique Key should be unique within slice",
    action="deduplicate_keep_first",
)

dup_exact_row = dfw.duplicated(keep=False)
log_check(
    "exact_duplicate_rows_in_slice",
    dup_exact_row,
    severity="warn",
    rule="Entire row should not duplicate",
    action="deduplicate",
)

pd.DataFrame(dq_records)

slice shape: (9862266, 57)


,check_name,severity,failed_rows,total_rows,failed_pct,rule,action,notes,run_ts
0,created_ts_parse_failed,error,0,9862266,0.0000,Created Date must parse to datetime,drop,,2026-05-06 12:37:23.217431+00:00
1,closed_ts_parse_failed_when_present,warn,0,9862266,0.0000,Closed Date must parse when non-empty,retain_open_flag_or_drop_if_invalid,,2026-05-06 12:37:27.167314+00:00
2,created_outside_2022_2025,info,0,9862266,0.0000,Created timestamp in 2022-01-01..2025-12-31,drop,,2026-05-06 12:37:27.366464+00:00
3,negative_resolution_duration,error,13441,9862266,0.1363,closed_ts >= created_ts for closed cases,drop,,2026-05-06 12:38:22.554752+00:00
4,implausible_closed_ts,error,9,9862266,0.0001,"closed_ts in [2020-01-01, today+30d]",drop,,2026-05-06 12:38:22.597916+00:00
5,duplicate_unique_key_in_slice,error,0,9862266,0.0000,Unique Key should be unique within slice,deduplicate_keep_first,,2026-05-06 12:38:29.624828+00:00
6,exact_duplicate_rows_in_slice,warn,0,9862266,0.0000,Entire row should not duplicate,deduplicate,,2026-05-06 12:39:51.842880+00:00


## 7) Missingness in Cleaning-Critical Fields

Measure null/blank rates for fields required to produce a clean, consistent output dataset.


In [11]:
key_fields = ["agency", "complaint_type", "borough", "open_data_channel_type", "status"]
for c in key_fields:
    mask = dfw[c].isna() | dfw[c].astype("string").str.strip().eq("")
    log_check(
        f"missing_{c}",
        mask,
        severity="warn",
        rule=f"{c} should be present for cleaning completeness",
        action="fill_UNKNOWN_or_keep_with_flag",
    )

missing_summary = (
    dfw[key_fields]
    .isna()
    .mean()
    .mul(100)
    .round(2)
    .rename("missing_pct")
    .reset_index()
    .rename(columns={"index": "field"})
)
missing_summary

,field,missing_pct
0,agency,0.0
1,complaint_type,0.0
2,borough,0.0
3,open_data_channel_type,0.0
4,status,0.0


## 8) Cleaning Validation Snapshot

Provide a compact, non-visual check of missingness concentration to support cleaning decisions.


In [12]:
tmp = dfw.assign(
    complaint_missing=dfw["complaint_type"].isna()
    | dfw["complaint_type"].astype("string").str.strip().eq(""),
    borough_missing=dfw["borough"].isna()
    | dfw["borough"].astype("string").str.strip().eq(""),
)

by_agency = (
    tmp.groupby("agency", dropna=False)[["complaint_missing", "borough_missing"]]
    .mean()
    .mul(100)
    .round(2)
    .sort_values("complaint_missing", ascending=False)
    .head(20)
)

by_agency

,complaint_missing,borough_missing
agency,,
3-1-1,0.0,0.0
DPR,0.0,0.0
OTI,0.0,0.0
OSE,0.0,0.0
NYPD,0.0,0.0
HPD,0.0,0.0
EDC,0.0,0.0
DSNY,0.0,0.0
DOT,0.0,0.0


## 10) Apply Cleaning Rules

Apply standardized cleaning actions:

- normalize key text fields
- drop invalid time records
- deduplicate by `unique_key` then full row
- recompute duration metrics


In [13]:
clean = dfw.copy()

clean["agency"] = clean["agency"].astype("string").str.strip().replace({"": pd.NA})
clean["agency_missing"] = clean["agency"].isna()

for c in ["complaint_type", "borough", "open_data_channel_type", "status"]:
    clean[c] = clean[c].astype("string").str.strip()
    clean[c] = clean[c].replace({"": pd.NA}).fillna("UNKNOWN")

drop_mask = (
    clean["created_ts"].isna()
    | (clean["is_closed"] & (clean["closed_ts"] < clean["created_ts"]))
    | (
        clean["is_closed"]
        & (
            (clean["closed_ts"] < pd.Timestamp("2020-01-01"))
            | (clean["closed_ts"] > PIPELINE_DATE + pd.Timedelta(days=30))
        )
    )
)

clean = clean.loc[~drop_mask].copy()
clean = clean.drop_duplicates(subset=["unique_key"], keep="first")
clean = clean.drop_duplicates(keep="first")

clean["resolution_minutes"] = (
    clean["closed_ts"] - clean["created_ts"]
).dt.total_seconds() / 60.0
clean["resolution_days"] = clean["resolution_minutes"] / (60 * 24)

print("clean shape:", clean.shape)
clean[["created_ts", "closed_ts", "resolution_days"]].head()

clean shape: (9848824, 61)


,created_ts,closed_ts,resolution_days
0,2022-01-01 00:00:00,2022-01-03 08:39:00,2.360417
1,2022-01-01 00:00:10,2022-01-01 01:00:11,0.041678
2,2022-01-01 00:00:57,2022-01-01 00:58:22,0.039873
3,2022-01-01 00:01:04,2022-01-01 00:18:49,0.012326
4,2022-01-01 00:01:17,2022-01-01 02:55:16,0.120822


## 10b) Final Column Projection

Restrict the parquet schema to the columns the SQL pipeline expects. This makes the parquet a contract with `notebook/sql/01_clean.sql` and downstream views.


In [14]:
keep_cols = [
    "unique_key",
    "created_ts",
    "closed_ts",
    "agency",
    "agency_missing",
    "complaint_type",
    "descriptor",
    "borough",
    "open_data_channel_type",
    "status",
    "incident_zip",
    "is_closed",
    "resolution_minutes",
    "resolution_days",
]
clean = clean[keep_cols]
clean.dtypes

unique_key                        object
created_ts                datetime64[ns]
closed_ts                 datetime64[ns]
agency                    string[python]
agency_missing                      bool
complaint_type            string[python]
descriptor                        object
borough                   string[python]
open_data_channel_type    string[python]
status                    string[python]
incident_zip                      object
is_closed                           bool
resolution_minutes               float64
resolution_days                  float64
dtype: object

## 11) Export Clean Dataset and Quality Log

Persist cleaned outputs and quality-audit artifacts for reproducibility and handoff.


In [16]:
dq_log = pd.DataFrame(dq_records).sort_values(
    ["severity", "failed_rows"], ascending=[True, False]
)
dq_log["run_id"] = RUN_ID
dq_log.to_csv(DQ_LOG_CSV, index=False)

# pandas.to_parquet can fail with some pandas/pyarrow combinations (ArrowKeyError).
# Write via pyarrow directly first, then fall back to pandas with pyarrow engine.
try:
    import pyarrow as pa
    import pyarrow.parquet as pq

    table = pa.Table.from_pandas(clean, preserve_index=False)
    pq.write_table(table, CLEAN_PARQUET)
except Exception:
    clean.to_parquet(CLEAN_PARQUET, index=False, engine="pyarrow")

summary = {
    "rows_clean": len(clean),
    "unique_keys": clean["unique_key"].nunique(),
    "created_min": clean["created_ts"].min(),
    "created_max": clean["created_ts"].max(),
    "neg_duration_rows": int(
        (clean["is_closed"] & (clean["resolution_minutes"] < 0)).sum()
    ),
    "missing_agency_pct": round(100 * clean["agency"].eq("UNKNOWN").mean(), 2),
    "missing_complaint_pct": round(
        100 * clean["complaint_type"].eq("UNKNOWN").mean(), 2
    ),
}

print("Saved:", CLEAN_PARQUET)
print("Saved:", DQ_LOG_CSV)
summary["run_id"] = RUN_ID
summary["pipeline_date"] = str(PIPELINE_DATE)
with open(SUMMARY_JSON, "w") as f:
    json.dump({k: str(v) for k, v in summary.items()}, f, indent=2)
print("Saved:", SUMMARY_JSON)
summary

Saved: /Users/tuanhung/Documents/GitHub/NYC311/data/processed/311_clean_2022_2025.parquet
Saved: /Users/tuanhung/Documents/GitHub/NYC311/data/processed/311_data_quality_log_2022_2025.csv
Saved: /Users/tuanhung/Documents/GitHub/NYC311/data/processed/pipeline_summary.json


{'rows_clean': 9848824,
 'unique_keys': 9848824,
 'created_min': Timestamp('2022-01-01 00:00:00'),
 'created_max': Timestamp('2025-01-01 23:44:39'),
 'neg_duration_rows': 0,
 'missing_agency_pct': np.float64(0.0),
 'missing_complaint_pct': np.float64(0.0),
 'run_id': '4b0bc154',
 'pipeline_date': '2026-05-02 00:00:00'}